# 03 — DPO and the Prospective Memory Policy

This notebook covers:
1. Why SFT alone isn't enough for Trace's "witness, not coach" requirement
2. The DPO (Direct Preference Optimization) training objective
3. What chosen vs. rejected patterns look like in this dataset
4. How preference learning shapes the prospective-memory policy

**Status:** DPO training is a scaffold in this repo. This notebook explains
the design rationale so the data and policy goals are legible before
the training loop is live.

## 1. The limitation of SFT for policy behavior

SFT teaches the model *what the correct output looks like* on training examples.
This is effective for structured extraction — it learns the schema, the tier
definitions, and the JSON format.

But SFT can't easily teach *what to avoid*. Two failure modes matter here:

**Failure mode 1: Over-flattening commitments into semantic descriptions.**
Given: *"I've been meaning to reach out to Claire... I really should do it this week."*
A model without preference training may produce a clean `semantic` record
describing the person as someone who "values mentorship" — losing the forward
commitment signal entirely.

**Failure mode 2: Coaching / prescriptive responses.**
Given: *"Missed my third workout in a row."*
A model trained to be helpful may add a `stated_intent` like "Restart workout routine"
or add notes like "Consider scheduling exercise in advance." Neither was stated
by the person — the model invented them.

DPO addresses both by training the model to *prefer correct outputs* and
*avoid incorrect ones* using paired examples.

## 2. DPO objective (intuition)

DPO trains a language model to assign higher probability to **chosen** outputs
than to **rejected** outputs for each prompt, while staying close to the
reference policy (the SFT checkpoint).

The loss for one preference pair `(prompt, chosen, rejected)` is:

```
L = -log σ(β * (log π(chosen|prompt) - log π_ref(chosen|prompt)
                - log π(rejected|prompt) + log π_ref(rejected|prompt)))
```

Where:
- `π` is the policy being trained
- `π_ref` is the reference policy (SFT checkpoint — frozen)
- `β` controls the KL penalty — how far `π` can diverge from `π_ref`
- `σ` is the sigmoid function

**What β does in practice:**
- Low β (0.05–0.1): aggressive policy shaping; higher risk of collapse
- High β (0.5–1.0): conservative; smaller deviation from reference
- We use β=0.1 — standard starting point for a narrow behavioral constraint

**The Trace-specific goal:**
The reference policy already knows how to produce structured Trace records (SFT).
DPO refines *which* records it prefers — specifically, records that correctly
identify prospective commitments without coaching or prescribing action.

In [ ]:
import json
import pathlib

DATA_DIR = pathlib.Path("../data")

def load_jsonl(path):
    records = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records

dpo_pairs = load_jsonl(DATA_DIR / "sample_dpo.jsonl")
print(f"DPO preference pairs: {len(dpo_pairs)}")

## 3. Inspecting chosen vs. rejected patterns

In [ ]:
# Display each preference pair side by side
for i, pair in enumerate(dpo_pairs):
    input_text = pair["prompt"].split("Input:")[-1].strip()
    chosen = json.loads(pair["chosen"])
    rejected = json.loads(pair["rejected"])

    print(f"\n{'═'*60}")
    print(f"Pair {i+1}")
    print(f"{'═'*60}")
    print(f"INPUT: {input_text[:120]}..." if len(input_text) > 120 else f"INPUT: {input_text}")
    print()
    print("CHOSEN (correct):")
    print(f"  tier:    {chosen.get('memory_tier')}")
    print(f"  intent:  {chosen.get('stated_intent')}")
    print(f"  summary: {chosen.get('content_summary', '')[:100]}")
    print()
    print("REJECTED (incorrect):")
    print(f"  tier:    {rejected.get('memory_tier')}")
    print(f"  intent:  {rejected.get('stated_intent')}")
    print(f"  summary: {rejected.get('content_summary', '')[:100]}")
    print()
    
    # Identify what type of error the rejected response makes
    chosen_tier = chosen.get('memory_tier')
    rejected_tier = rejected.get('memory_tier')
    chosen_intent = chosen.get('stated_intent')
    rejected_intent = rejected.get('stated_intent')
    
    if chosen_tier != rejected_tier:
        print(f"  ⚠ TIER ERROR: chosen={chosen_tier} vs rejected={rejected_tier}")
    if chosen_intent and not rejected_intent:
        print(f"  ⚠ INTENT DROPPED: chosen has intent, rejected discards it")
    if not chosen_intent and rejected_intent:
        print(f"  ⚠ INTENT HALLUCINATED: rejected invents a stated_intent")
    if rejected.get('content_summary', '').count('should') > 0:
        print(f"  ⚠ COACHING: rejected summary contains prescriptive language")

## 4. "Witness, not coach" as a preference-learning target

This constraint is more than a style preference — it's a first-class design
requirement for Trace. Here's why it matters and how DPO enforces it.

### The witness principle

Trace is a *memory system*, not an assistant. Its job is to accurately record
and structure what a person actually said, intended, and did — not to evaluate
it, improve it, or suggest alternatives.

A model that adds `"stated_intent": "Restart workout routine"` when the person
only said *"missed another workout"* is not a witness — it's a coach. This
corrupts the memory record. Later intention-reality tracking would compare a
real outcome against an intent the person never had.

### What the rejected patterns signal

The three rejected responses in `sample_dpo.jsonl` each represent a distinct
failure mode:

| Pair | Failure mode | Effect on memory |
|------|-------------|------------------|
| 1 | Over-flattens prospective signal → `semantic` | Prospective commitment lost from the record |
| 2 | Invents a `stated_intent`; adds coaching framing | Fabricated intent poisons future tracking |
| 3 | Classifies explicit commitment as `episodic` | Social accountability signal lost |

### What DPO learns

By seeing these pairs, the model learns:
- When the input contains a forward commitment signal → prefer `prospective`
- When no intent was stated → prefer `null`, not a fabricated string
- When summarizing → prefer neutral observation over prescriptive framing

The KL penalty (β) ensures this doesn't override the schema and extraction
quality learned during SFT.

In [ ]:
# Summarize the DPO dataset policy signals
print("DPO preference pair policy signals:")
print()

policy_signals = [
    {
        "pair": 1,
        "input_type": "Weak prospective signal (repeated deferral + 'I should')",
        "chosen_teaches": "Prospective tier with weak-but-present commitment",
        "rejected_teaches": "Do not flatten commitment into generic semantic self-description",
    },
    {
        "pair": 2,
        "input_type": "Episodic pattern with no stated intent",
        "chosen_teaches": "Record what happened; stated_intent=null",
        "rejected_teaches": "Do not invent intents or add coaching in the summary",
    },
    {
        "pair": 3,
        "input_type": "Explicit commitment with social accountability",
        "chosen_teaches": "Prospective tier; extract stated intent directly",
        "rejected_teaches": "Do not reduce explicit commitments to episodic observations",
    },
]

for p in policy_signals:
    print(f"  Pair {p['pair']}: {p['input_type']}")
    print(f"    ✓ Chosen teaches:  {p['chosen_teaches']}")
    print(f"    ✗ Rejected shows:  {p['rejected_teaches']}")
    print()

## 5. Next steps for Stage 3

When MLX-LM DPO support is confirmed available:

1. **Expand the DPO dataset** — the 3 synthetic pairs here are a conceptual
   starter. A meaningful DPO run typically needs 50–500 pairs to have a stable
   training signal. Generate more pairs from real traces in `data/raw_private/`.

2. **Start from the SFT adapter** — set `adapter_path` in
   `configs/dpo_trace_qwen25_3b.yaml` to `outputs/sft_qwen25_3b`.
   DPO starting from the SFT policy diverges meaningfully from base.

3. **Run with β=0.1 first** — then tune based on eval results:
   - If the model starts fabricating intents: increase β (more conservative)
   - If the model still coaches: decrease β (more aggressive shaping)

4. **Eval the DPO checkpoint** with `eval_extraction.py` — comparing
   SFT vs DPO adapter on `eval_gold.jsonl`, especially on `aspiration_vs_commitment`
   tagged examples.

```bash
# SFT adapter eval baseline
python scripts/eval_extraction.py \
    --config configs/sft_trace_qwen25_3b.yaml \
    --adapter outputs/sft_qwen25_3b

# DPO adapter eval (after training)
python scripts/eval_extraction.py \
    --config configs/sft_trace_qwen25_3b.yaml \
    --adapter outputs/dpo_qwen25_3b
```